# Adaptive filter comparison and diagnostics

同一個 2-tap plant 上比較 LMS families、NLMS controller、Newton、APA 和 DCT/DFT-LMS，保留分析與係數比較。

來源：`doctest.ipynb`。所有資料與結果留在記憶體，圖表使用共用深色風格。

執行順序：設定 → 參數與資料 → 模型與訓練 → 圖表與數值摘要。

## 1. Setup

In [ ]:
from pathlib import Path
import sys
from functools import partial

import numpy as np

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "adsp").is_dir())
if str(project_root) not in sys.path:
  sys.path.insert(0, str(project_root))

from adsp import data, training, statistics as stats
from adsp import plotting as plot
from adsp import filters as af
from adsp import equalization as ae
from adsp.datasets import SampleBatch

plot.use_dark()

## 2. Shared plant data and factories

In [ ]:
n_samples, n_trials, seed = 300, 1, 1
h, noise_var = np.array([0.5, -0.3]), 1e-4
make_data = partial(data.system_identification, n_samples, h=h, noise_var=noise_var)
models = {
  "LMS": partial(af.LMS, filter_order=1, mu=0.05),
  "NLMS": partial(af.NLMS, filter_order=1, mu=0.5),
  "Controlled NLMS":
  lambda: af.NLMS(filter_order=1, mu=0.5, controller=af.NLMSController(sigma_v2=noise_var, mu_max=1.0)),
  "LMS-Newton": partial(af.LMSNewton, filter_order=1, mu=0.02, alpha=0.5, delta=1.0),
  "APA P=1": partial(af.AffineProjection, filter_order=1, mu=0.5, P=1),
  "APA P=2": partial(af.AffineProjection, filter_order=1, mu=0.5, P=2),
  "Sign-Error LMS": partial(af.SignErrorLMS, filter_order=1, mu=0.02),
  "Sign-Data LMS": partial(af.SignDataLMS, filter_order=1, mu=0.02),
  "Sign-Sign LMS": partial(af.SignSignLMS, filter_order=1, mu=0.02),
  "Dual-Sign LMS": partial(af.DualSignLMS, filter_order=1, mu=0.02),
  "Power-of-Two LMS": partial(af.PowerOfTwoErrorLMS, filter_order=1, mu=0.02),
  "DCT-LMS": partial(af.TransformDomain, filter_order=1, mu=0.02, matrix="dct"),
  "DFT-LMS": partial(af.TransformDomain, filter_order=1, mu=0.02, matrix="dft"),
}
results = training.compare(models, make_data, n_trials=n_trials, seed=seed)

## 3. Physical MSE/MSD and learned taps

In [ ]:
fig, grid = plot.axes(rows=2)
plot.learning(results, ax=grid[0, 0], noise_floor=noise_var, window=11)
plot.learning(results, quantity="msd", ax=grid[1, 0], window=11, title="True direct-form coefficient MSD")
plot.show(fig)
plot.coefficients(results["LMS"], reference=np.conj(h))
plot.show()
training.summary(results, tail=100)
print("LMS theoretical analysis:", models["LMS"]().analyze(np.eye(2)))